# Autoresearch — Free GPU Edition (Colab T4)

Run Andrej Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) autonomously on a **free** Colab T4 GPU with **free** LLM APIs.

**Zero cost**: Free GPU (Colab T4) + Free LLM APIs (OpenRouter/Groq/Gemini) with daisy-chain failover.

## Setup
1. **GPU**: Make sure you selected `Runtime > Change runtime type > T4 GPU`
2. **API Keys**: Add at least one free API key in `Secrets` (key icon, left sidebar):
   - `OPENROUTER_API_KEY` — Get free at [openrouter.ai](https://openrouter.ai) (recommended)
   - `GROQ_API_KEY` — Get free at [console.groq.com](https://console.groq.com)
   - `GEMINI_API_KEY` — Get free at [aistudio.google.com](https://aistudio.google.com/apikey)
3. **Run all cells** — the notebook handles everything else automatically

The more API keys you add, the more resilient the daisy-chain becomes against rate limits.

In [ ]:
#@title 1. Install Dependencies
!pip install -q openai tiktoken rustbpe pyarrow requests 2>/dev/null
# Try installing kernels (needed for Ampere+ GPUs, optional on T4)
!pip install -q kernels 2>/dev/null || echo '[autoresearch] kernels package not available, using SDPA (fine for T4)'
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Go to Runtime > Change runtime type > T4 GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

In [ ]:
#@title 2. Mount Google Drive for Persistence
import os
from google.colab import drive

drive.mount('/content/drive')

PERSIST_DIR = '/content/drive/MyDrive/autoresearch'
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs(os.path.join(PERSIST_DIR, 'cache'), exist_ok=True)

# Symlink cache so prepare.py stores data on Drive (survives disconnects)
cache_link = os.path.expanduser('~/.cache/autoresearch')
cache_target = os.path.join(PERSIST_DIR, 'cache')
if os.path.islink(cache_link):
    os.unlink(cache_link)
elif os.path.exists(cache_link):
    import shutil
    shutil.rmtree(cache_link)
os.makedirs(os.path.dirname(cache_link), exist_ok=True)
os.symlink(cache_target, cache_link)
print(f'Cache linked: {cache_link} -> {cache_target}')
print(f'Persistence dir: {PERSIST_DIR}')

In [ ]:
#@title 3. Clone Repo & Prepare Data
REPO_URL = 'https://github.com/AI-ANK/autoresearch_colab.git'  #@param {type:"string"}
REPO_DIR = '/content/autoresearch_colab'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Check if data already exists on Drive (from a previous session)
data_dir = os.path.expanduser('~/.cache/autoresearch/data')
if os.path.exists(data_dir) and len(os.listdir(data_dir)) > 2:
    print(f'Data already cached on Drive ({len(os.listdir(data_dir))} files). Skipping download.')
else:
    print('Downloading data shards and training tokenizer (one-time, ~2-3 min)...')
    !python prepare.py --num-shards 4

In [ ]:
#@title 4. Configure API Keys (Daisy-Chain LLM)
from openai import OpenAI
import time as _time
import traceback

# Load API keys from Colab Secrets
from google.colab import userdata

def _safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

OPENROUTER_KEY = _safe_get_secret('OPENROUTER_API_KEY')
GROQ_KEY = _safe_get_secret('GROQ_API_KEY')
GEMINI_KEY = _safe_get_secret('GEMINI_API_KEY')


class DaisyChainLLM:
    """Try free LLM APIs in sequence. If one fails/rate-limits, try next."""

    def __init__(self):
        self.providers = []
        if OPENROUTER_KEY:
            self.providers.append(("openrouter", OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=OPENROUTER_KEY
            ), "qwen/qwen3-coder:free"))
        if GROQ_KEY:
            self.providers.append(("groq", OpenAI(
                base_url="https://api.groq.com/openai/v1",
                api_key=GROQ_KEY
            ), "llama-3.3-70b-versatile"))
        if GEMINI_KEY:
            self.providers.append(("gemini", OpenAI(
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                api_key=GEMINI_KEY
            ), "gemini-2.5-flash"))

        assert len(self.providers) > 0, (
            "No API keys found! Add at least one of: "
            "OPENROUTER_API_KEY, GROQ_API_KEY, GEMINI_API_KEY "
            "in Colab Secrets (key icon, left sidebar)"
        )
        print(f"Daisy-chain LLM initialized with {len(self.providers)} providers: "
              f"{', '.join(name for name, _, _ in self.providers)}")

    def chat(self, messages, max_tokens=8000, temperature=0.7):
        """Try each provider in sequence until one succeeds."""
        errors = []
        for name, client, model in self.providers:
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    max_tokens=max_tokens,
                    temperature=temperature,
                )
                content = response.choices[0].message.content
                print(f"  [LLM: {name}/{model}]")
                return content
            except Exception as e:
                err_msg = str(e)[:200]
                errors.append(f"{name}: {err_msg}")
                print(f"  [{name}] failed: {err_msg}")
                # Rate limit: wait before trying next
                if '429' in str(e) or 'rate' in str(e).lower():
                    _time.sleep(5)
                continue
        raise RuntimeError(
            f"All LLM providers failed:\n" +
            "\n".join(f"  - {e}" for e in errors)
        )


llm = DaisyChainLLM()

In [ ]:
#@title 5. Anti-Disconnect Keep-Alive
from IPython.display import display, HTML

# Periodic JS reconnect to prevent Colab idle timeout
display(HTML('''
<script>
function keepAlive() {
    google.colab.kernel.invokeFunction('connect', [], {});
}
// Reconnect every 60 seconds
setInterval(keepAlive, 60000);
console.log('Keep-alive installed: reconnecting every 60s');
</script>
<div style="color: green; font-weight: bold;">Keep-alive active (reconnects every 60s)</div>
'''))

print('Anti-disconnect keep-alive installed.')
print('Tip: Keep this browser tab visible/focused for best results.')
print('Tip: If disconnected, just re-run this notebook — it resumes from Drive.')

In [ ]:
#@title 6. Quick Smoke Test (single training run)
# Run one training cycle to make sure everything works before going autonomous
import subprocess

print('Running smoke test (5 min training run)...')
result = subprocess.run(
    ['python', 'train.py'],
    capture_output=True, text=True, timeout=700,
    cwd=REPO_DIR
)

# Show output
output = result.stdout + result.stderr
# Extract summary
for line in output.split('\n'):
    if line.startswith(('val_bpb:', 'peak_vram_mb:', 'mfu_percent:',
                        'num_params_M:', 'depth:', 'total_tokens_M:', '---')):
        print(line)

if result.returncode != 0:
    print('\n--- SMOKE TEST FAILED ---')
    print('Last 30 lines of output:')
    print('\n'.join(output.split('\n')[-30:]))
    raise RuntimeError('Smoke test failed. Check output above.')
else:
    print('\nSmoke test passed!')

In [ ]:
#@title 7. Autonomous Research Loop
import subprocess
import re
import json
import shutil
import time

MAX_EXPERIMENTS = 200  #@param {type:"integer"}

# ---------------------------------------------------------------------------
# System prompt for the autonomous researcher
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = open(os.path.join(REPO_DIR, 'program_colab.md')).read()

# ---------------------------------------------------------------------------
# State management
# ---------------------------------------------------------------------------

RESULTS_FILE = os.path.join(REPO_DIR, 'results.tsv')
RESULTS_BACKUP = os.path.join(PERSIST_DIR, 'results.tsv')
TRAINPY_BACKUP = os.path.join(PERSIST_DIR, 'train.py.best')
STATE_FILE = os.path.join(PERSIST_DIR, 'agent_state.json')


def load_state():
    """Load state from Drive (for recovery after disconnect)."""
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return {'experiment_num': 0, 'best_bpb': None, 'conversation_summary': ''}


def save_state(state):
    """Save state to Drive for persistence across disconnects."""
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f)
    # Also backup results.tsv
    if os.path.exists(RESULTS_FILE):
        shutil.copy2(RESULTS_FILE, RESULTS_BACKUP)


def read_results_tsv():
    """Read results.tsv and return as string."""
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            return f.read()
    return 'commit\tval_bpb\tmemory_gb\tstatus\tdescription\n'


def append_result(commit, val_bpb, memory_gb, status, description):
    """Append a result to results.tsv."""
    if not os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'w') as f:
            f.write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')
    with open(RESULTS_FILE, 'a') as f:
        f.write(f'{commit}\t{val_bpb}\t{memory_gb}\t{status}\t{description}\n')


def get_short_commit():
    """Get current git commit hash (short)."""
    r = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                       capture_output=True, text=True, cwd=REPO_DIR)
    return r.stdout.strip()


def run_training():
    """Run train.py and return (val_bpb, peak_vram_mb, full_output) or None on crash."""
    try:
        result = subprocess.run(
            ['python', 'train.py'],
            capture_output=True, text=True, timeout=700,
            cwd=REPO_DIR
        )
        output = result.stdout + result.stderr

        # Save full log
        with open(os.path.join(REPO_DIR, 'run.log'), 'w') as f:
            f.write(output)

        if result.returncode != 0:
            return None, None, output

        # Parse results
        val_bpb = None
        peak_vram = None
        for line in output.split('\n'):
            if line.startswith('val_bpb:'):
                val_bpb = float(line.split()[-1])
            elif line.startswith('peak_vram_mb:'):
                peak_vram = float(line.split()[-1])

        return val_bpb, peak_vram, output

    except subprocess.TimeoutExpired:
        return None, None, 'TIMEOUT: Training exceeded 700s'
    except Exception as e:
        return None, None, f'ERROR: {e}'


def extract_code_block(text):
    """Extract Python code from markdown code block in LLM response."""
    # Look for ```python ... ``` blocks
    pattern = r'```python\s*\n(.*?)```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        # Return the longest match (likely the full file)
        return max(matches, key=len)
    # Fallback: try ``` ... ``` without language tag
    pattern = r'```\s*\n(.*?)```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return max(matches, key=len)
    return None


# ---------------------------------------------------------------------------
# Initialize git for experiment tracking
# ---------------------------------------------------------------------------

subprocess.run(['git', 'add', '-A'], cwd=REPO_DIR)
subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=REPO_DIR)
# Create initial commit if needed
r = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True, cwd=REPO_DIR)
if r.returncode != 0:
    subprocess.run(['git', 'config', 'user.email', 'autoresearch@colab'], cwd=REPO_DIR)
    subprocess.run(['git', 'config', 'user.name', 'autoresearch'], cwd=REPO_DIR)
    subprocess.run(['git', 'add', '-A'], cwd=REPO_DIR)
    subprocess.run(['git', 'commit', '-m', 'initial'], cwd=REPO_DIR)

# Create experiment branch
import datetime
tag = datetime.datetime.now().strftime('%b%d').lower()
branch = f'autoresearch/{tag}'
subprocess.run(['git', 'checkout', '-b', branch], cwd=REPO_DIR,
               capture_output=True)  # ignore if exists

# ---------------------------------------------------------------------------
# Recovery: load state from Drive if resuming after disconnect
# ---------------------------------------------------------------------------

state = load_state()
start_experiment = state['experiment_num']

if start_experiment > 0:
    print(f'Resuming from experiment #{start_experiment}')
    if os.path.exists(RESULTS_BACKUP) and not os.path.exists(RESULTS_FILE):
        shutil.copy2(RESULTS_BACKUP, RESULTS_FILE)
        print(f'Restored results.tsv from Drive')
else:
    # Initialize results.tsv
    if not os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'w') as f:
            f.write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')

# ---------------------------------------------------------------------------
# The autonomous experiment loop
# ---------------------------------------------------------------------------

# Read current train.py
with open(os.path.join(REPO_DIR, 'train.py')) as f:
    current_code = f.read()

# Build initial conversation
conversation = []

# Include results history if resuming
results_context = read_results_tsv()
if start_experiment == 0:
    user_msg = (
        f"Here is the current train.py:\n\n```python\n{current_code}\n```\n\n"
        f"Run the baseline first (no changes). Then start experimenting.\n"
        f"For each experiment, respond with:\n"
        f"1. A short description of what you're trying (1 sentence)\n"
        f"2. The COMPLETE modified train.py in a ```python code block\n\n"
        f"If this is the baseline run, just say 'BASELINE' and return the code unchanged."
    )
else:
    user_msg = (
        f"We are resuming after a disconnect. Here are the results so far:\n\n"
        f"```\n{results_context}```\n\n"
        f"Here is the current train.py:\n\n```python\n{current_code}\n```\n\n"
        f"{state.get('conversation_summary', '')}\n\n"
        f"Continue experimenting. Respond with:\n"
        f"1. A short description of what you're trying\n"
        f"2. The COMPLETE modified train.py in a ```python code block"
    )

conversation.append({"role": "user", "content": user_msg})

print(f'Starting autonomous research loop (experiments {start_experiment} to {MAX_EXPERIMENTS})')
print(f'Each experiment takes ~5-6 minutes. {MAX_EXPERIMENTS - start_experiment} experiments remaining.')
print('=' * 70)

for exp_num in range(start_experiment, MAX_EXPERIMENTS):
    print(f'\n{"="*70}')
    print(f'EXPERIMENT #{exp_num}')
    print(f'{"="*70}')

    # 1. Ask LLM for next experiment
    print('Asking LLM for next experiment...')
    try:
        response = llm.chat(
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + conversation,
            max_tokens=12000,
            temperature=0.7,
        )
    except RuntimeError as e:
        print(f'All LLM providers failed: {e}')
        print('Waiting 60s before retrying...')
        time.sleep(60)
        continue

    # 2. Extract description and code
    # Get description (first non-empty line that isn't code)
    description = 'unknown experiment'
    for line in response.split('\n'):
        line = line.strip()
        if line and not line.startswith('```') and not line.startswith('import') and len(line) > 5:
            description = line.lstrip('#').lstrip('*').strip()[:100]
            break

    is_baseline = 'baseline' in response.lower()[:200] and exp_num == 0
    if is_baseline:
        description = 'baseline'

    print(f'Experiment: {description}')

    # 3. Extract and write modified train.py
    new_code = extract_code_block(response)

    if new_code and len(new_code) > 100:
        with open(os.path.join(REPO_DIR, 'train.py'), 'w') as f:
            f.write(new_code)
    elif not is_baseline:
        print('WARNING: Could not extract code from LLM response. Using current code.')
        description = f'(code extraction failed) {description}'

    # 4. Git commit
    subprocess.run(['git', 'add', 'train.py'], cwd=REPO_DIR)
    subprocess.run(['git', 'commit', '-m', f'exp{exp_num}: {description[:60]}'],
                   cwd=REPO_DIR, capture_output=True)
    commit = get_short_commit()

    # 5. Run training
    print(f'Running training (5 min)...')
    t_start = time.time()
    val_bpb, peak_vram, output = run_training()
    t_elapsed = time.time() - t_start
    print(f'Training completed in {t_elapsed:.0f}s')

    # 6. Process results
    if val_bpb is None:
        # Crash
        print(f'CRASH!')
        last_lines = '\n'.join(output.split('\n')[-20:])
        print(f'Last 20 lines:\n{last_lines}')
        append_result(commit, '0.000000', '0.0', 'crash', description)
        # Revert
        subprocess.run(['git', 'reset', '--hard', 'HEAD~1'], cwd=REPO_DIR, capture_output=True)
        # Read current code back
        with open(os.path.join(REPO_DIR, 'train.py')) as f:
            current_code = f.read()
        feedback = (
            f"Experiment #{exp_num} CRASHED. Error:\n"
            f"```\n{last_lines}\n```\n\n"
            f"Reverted to previous code. Current train.py:\n"
            f"```python\n{current_code}\n```\n\n"
            f"Try a different approach. Respond with description + complete train.py."
        )
    else:
        memory_gb = peak_vram / 1024 if peak_vram else 0
        print(f'val_bpb: {val_bpb:.6f} | peak_vram: {memory_gb:.1f} GB')

        # Decide keep/discard
        if state['best_bpb'] is None or val_bpb < state['best_bpb']:
            status = 'keep'
            improvement = 0 if state['best_bpb'] is None else state['best_bpb'] - val_bpb
            state['best_bpb'] = val_bpb
            print(f'KEEP! (improvement: {improvement:.6f})')
            # Backup best train.py to Drive
            shutil.copy2(os.path.join(REPO_DIR, 'train.py'), TRAINPY_BACKUP)
            # Read current code
            with open(os.path.join(REPO_DIR, 'train.py')) as f:
                current_code = f.read()
        else:
            status = 'discard'
            print(f'DISCARD (best remains {state["best_bpb"]:.6f})')
            # Revert
            subprocess.run(['git', 'reset', '--hard', 'HEAD~1'], cwd=REPO_DIR, capture_output=True)
            with open(os.path.join(REPO_DIR, 'train.py')) as f:
                current_code = f.read()

        append_result(commit, f'{val_bpb:.6f}', f'{memory_gb:.1f}', status, description)

        feedback = (
            f"Experiment #{exp_num} result: val_bpb={val_bpb:.6f}, "
            f"peak_vram={memory_gb:.1f}GB, status={status}\n"
            f"Best val_bpb so far: {state['best_bpb']:.6f}\n\n"
            f"Full results history:\n```\n{read_results_tsv()}```\n\n"
            f"Current train.py:\n```python\n{current_code}\n```\n\n"
            f"Propose the next experiment. Respond with description + complete train.py."
        )

    # 7. Update conversation (sliding window: keep last 6 exchanges)
    conversation.append({"role": "assistant", "content": response})
    conversation.append({"role": "user", "content": feedback})

    # Trim conversation to avoid context overflow
    if len(conversation) > 12:
        # Keep first message (has code) and last 10 messages
        conversation = [conversation[0]] + conversation[-10:]

    # 8. Save state to Drive
    state['experiment_num'] = exp_num + 1
    state['conversation_summary'] = (
        f"Completed {exp_num + 1} experiments. "
        f"Best val_bpb: {state['best_bpb']}"
    )
    save_state(state)

print('\n' + '=' * 70)
print(f'DONE! Completed {MAX_EXPERIMENTS} experiments.')
print(f'Best val_bpb: {state["best_bpb"]:.6f}')
print(f'Results saved to: {RESULTS_BACKUP}')
print(f'Best train.py saved to: {TRAINPY_BACKUP}')
print(f'\nFull results:\n{read_results_tsv()}')

In [ ]:
#@title 8. Recovery Cell (run after disconnect)
# If you got disconnected, just re-run cells 1-5 then this cell to resume.
# It will pick up from where it left off using state saved on Drive.

import json, os
STATE_FILE = '/content/drive/MyDrive/autoresearch/agent_state.json'
RESULTS_BACKUP = '/content/drive/MyDrive/autoresearch/results.tsv'

if os.path.exists(STATE_FILE):
    with open(STATE_FILE) as f:
        state = json.load(f)
    print(f"Last state: experiment #{state['experiment_num']}, best_bpb={state.get('best_bpb')}")
    print(f"Summary: {state.get('conversation_summary', 'N/A')}")
    if os.path.exists(RESULTS_BACKUP):
        with open(RESULTS_BACKUP) as f:
            print(f'\nResults from Drive:\n{f.read()}')
    print('\nTo resume: re-run Cell 7 (Autonomous Research Loop).')
else:
    print('No previous state found. This is a fresh start.')